# AI Health Monitoring Guide — History-Aware Risk Label Prediction using Medical-Guideline Features and LightGBM

This notebook uses your uploaded Excel dataset and builds a model where:
- **target variable** = `Risk Label`
- the model uses **today's input** + **3-day and 5-day history averages**
- it also uses **medical-guideline-based engineered features** for glucose, blood pressure, and cholesterol
- data is split **patient-wise** into **70% train / 20% validation / 10% test**
- metrics are reported separately for **train**, **validation**, and **test**


In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from lightgbm import LGBMClassifier


## 1. Load Dataset

In [ ]:
FILE_PATH = Path("patient_dataset_with_risk_labels.xlsx")
SHEET_NAME = "Daily Readings + Risk Label"

df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME, header=1)
df.head()


## 2. Basic Cleaning

In [ ]:
df = df.copy()
df.drop_duplicates(inplace=True)
df.columns = [str(c).strip() for c in df.columns]

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype(str).str.strip()

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df.sort_values(["Patient ID", "Date"], inplace=True)
df.reset_index(drop=True, inplace=True)

print(df.shape)
df.head()


## 3. Target Distribution

In [ ]:
df["Risk Label"].value_counts()


## 4. Medical-Guideline-Based Feature Engineering

In [ ]:
# Blood pressure categories
df["bp_stage"] = 0
df.loc[(df["Systolic BP (mmHg)"] >= 120) & (df["Systolic BP (mmHg)"] <= 129) & (df["Diastolic BP (mmHg)"] < 80), "bp_stage"] = 1
df.loc[((df["Systolic BP (mmHg)"] >= 130) & (df["Systolic BP (mmHg)"] <= 139)) | ((df["Diastolic BP (mmHg)"] >= 80) & (df["Diastolic BP (mmHg)"] <= 89)), "bp_stage"] = 2
df.loc[(df["Systolic BP (mmHg)"] >= 140) | (df["Diastolic BP (mmHg)"] >= 90), "bp_stage"] = 3
df.loc[(df["Systolic BP (mmHg)"] > 180) | (df["Diastolic BP (mmHg)"] > 120), "bp_stage"] = 4

# Glucose bands
df["glucose_band"] = 0
df.loc[(df["Blood Glucose (mg/dL)"] >= 100) & (df["Blood Glucose (mg/dL)"] <= 125), "glucose_band"] = 1
df.loc[(df["Blood Glucose (mg/dL)"] >= 126) & (df["Blood Glucose (mg/dL)"] < 200), "glucose_band"] = 2
df.loc[df["Blood Glucose (mg/dL)"] >= 200, "glucose_band"] = 3

# Cholesterol bands
df["cholesterol_band"] = 0
df.loc[(df["Cholesterol (mg/dL)"] >= 200) & (df["Cholesterol (mg/dL)"] <= 239), "cholesterol_band"] = 1
df.loc[df["Cholesterol (mg/dL)"] >= 240, "cholesterol_band"] = 2

# Comorbidity burden
df["comorbidity_count"] = df[["Diabetes", "Hypertension", "Heart Disease"]].sum(axis=1)

# Guideline-style summary score
df["rule_risk_score"] = (
    df["bp_stage"] +
    df["glucose_band"] +
    df["cholesterol_band"] +
    df["comorbidity_count"]
)

df[["bp_stage", "glucose_band", "cholesterol_band", "comorbidity_count", "rule_risk_score"]].head()


## 5. History-Aware Features (3-day and 5-day averages)

In [ ]:
g = df.groupby("Patient ID", group_keys=False)

for col in ["Blood Glucose (mg/dL)", "Systolic BP (mmHg)", "Diastolic BP (mmHg)", "Pulse (bpm)", "Cholesterol (mg/dL)"]:
    safe_col = col.replace(" (mg/dL)", "").replace(" (mmHg)", "").replace(" (bpm)", "").replace(" ", "_").replace("/", "_")
    df[f"{safe_col}_avg_3d_prev"] = g[col].transform(lambda s: s.shift(1).rolling(3, min_periods=3).mean())
    df[f"{safe_col}_avg_5d_prev"] = g[col].transform(lambda s: s.shift(1).rolling(5, min_periods=5).mean())
    df[f"{safe_col}_yesterday"] = g[col].shift(1)

df["high_glucose_days_5d"] = g["Blood Glucose (mg/dL)"].transform(lambda s: (s.shift(1).ge(126)).rolling(5, min_periods=5).sum())
df["high_bp_days_5d"] = g["Systolic BP (mmHg)"].transform(lambda s: (s.shift(1).ge(140)).rolling(5, min_periods=5).sum())
df["high_cholesterol_days_5d"] = g["Cholesterol (mg/dL)"].transform(lambda s: (s.shift(1).ge(240)).rolling(5, min_periods=5).sum())

df["glucose_diff_vs_3d_avg"] = df["Blood Glucose (mg/dL)"] - df["Blood_Glucose_avg_3d_prev"]
df["glucose_diff_vs_yesterday"] = df["Blood Glucose (mg/dL)"] - df["Blood_Glucose_yesterday"]

df["systolic_diff_vs_3d_avg"] = df["Systolic BP (mmHg)"] - df["Systolic_BP_avg_3d_prev"]
df["systolic_diff_vs_yesterday"] = df["Systolic BP (mmHg)"] - df["Systolic_BP_yesterday"]

df["cholesterol_diff_vs_3d_avg"] = df["Cholesterol (mg/dL)"] - df["Cholesterol_avg_3d_prev"]
df["cholesterol_diff_vs_yesterday"] = df["Cholesterol (mg/dL)"] - df["Cholesterol_yesterday"]

model_df = df.dropna(subset=[
    "Blood_Glucose_avg_3d_prev", "Blood_Glucose_avg_5d_prev",
    "Systolic_BP_avg_3d_prev", "Systolic_BP_avg_5d_prev",
    "Diastolic_BP_avg_3d_prev", "Diastolic_BP_avg_5d_prev",
    "Pulse_avg_3d_prev", "Pulse_avg_5d_prev",
    "Cholesterol_avg_3d_prev", "Cholesterol_avg_5d_prev",
    "glucose_diff_vs_3d_avg", "glucose_diff_vs_yesterday",
    "systolic_diff_vs_3d_avg", "systolic_diff_vs_yesterday",
    "cholesterol_diff_vs_3d_avg", "cholesterol_diff_vs_yesterday",
    "high_glucose_days_5d", "high_bp_days_5d", "high_cholesterol_days_5d"
]).copy()

print(model_df.shape)
model_df.head()


## 6. Build Input Features and Encode Target

In [ ]:
feature_cols = [
    "Day", "Age", "Gender", "BMI",
    "Blood Glucose (mg/dL)", "Systolic BP (mmHg)", "Diastolic BP (mmHg)",
    "Pulse (bpm)", "Cholesterol (mg/dL)",
    "Diabetes", "Hypertension", "Heart Disease",
    "bp_stage", "glucose_band", "cholesterol_band", "comorbidity_count", "rule_risk_score",
    "Blood_Glucose_avg_3d_prev", "Blood_Glucose_avg_5d_prev",
    "Systolic_BP_avg_3d_prev", "Systolic_BP_avg_5d_prev",
    "Diastolic_BP_avg_3d_prev", "Diastolic_BP_avg_5d_prev",
    "Pulse_avg_3d_prev", "Pulse_avg_5d_prev",
    "Cholesterol_avg_3d_prev", "Cholesterol_avg_5d_prev",
    "glucose_diff_vs_3d_avg", "glucose_diff_vs_yesterday",
    "systolic_diff_vs_3d_avg", "systolic_diff_vs_yesterday",
    "cholesterol_diff_vs_3d_avg", "cholesterol_diff_vs_yesterday",
    "high_glucose_days_5d", "high_bp_days_5d", "high_cholesterol_days_5d"
]

gender_le = LabelEncoder()
model_df["Gender"] = gender_le.fit_transform(model_df["Gender"])

X = model_df[feature_cols].copy()
y_raw = model_df["Risk Label"].copy()
groups = model_df["Patient ID"].copy()

target_le = LabelEncoder()
y = target_le.fit_transform(y_raw)

print("Classes:", list(target_le.classes_))
X.head()


## 7. 70 / 20 / 10 Patient-wise Split

In [ ]:
gss1 = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=42)
train_idx, temp_idx = next(gss1.split(X, y, groups=groups))

X_train = X.iloc[train_idx].copy()
y_train = y[train_idx]
groups_train = groups.iloc[train_idx]

X_temp = X.iloc[temp_idx].copy()
y_temp = y[temp_idx]
groups_temp = groups.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, train_size=(20/30), random_state=42)
val_idx_rel, test_idx_rel = next(gss2.split(X_temp, y_temp, groups=groups_temp))

X_val = X_temp.iloc[val_idx_rel].copy()
y_val = y_temp[val_idx_rel]
groups_val = groups_temp.iloc[val_idx_rel]

X_test = X_temp.iloc[test_idx_rel].copy()
y_test = y_temp[test_idx_rel]
groups_test = groups_temp.iloc[test_idx_rel]

print("Train rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Test rows:", len(X_test))
print("Train patients:", groups_train.nunique())
print("Validation patients:", groups_val.nunique())
print("Test patients:", groups_test.nunique())


## 8. Train LightGBM Model

In [ ]:
lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.9,
    colsample_bytree=0.9,
    class_weight="balanced",
    random_state=42,
    objective="multiclass",
    verbose=-1
)

lgbm_model.fit(X_train, y_train)
print("LightGBM model trained successfully.")


## 9. Training Metrics

In [ ]:
y_train_pred = lgbm_model.predict(X_train)

print("Training Accuracy:", accuracy_score(y_train, y_train_pred))
print("Training Weighted Precision:", precision_score(y_train, y_train_pred, average="weighted", zero_division=0))
print("Training Weighted Recall:", recall_score(y_train, y_train_pred, average="weighted", zero_division=0))
print("Training Weighted F1:", f1_score(y_train, y_train_pred, average="weighted", zero_division=0))
print("\nTraining Classification Report:\n")
print(classification_report(y_train, y_train_pred, target_names=target_le.classes_))
print("Training Confusion Matrix:\n", confusion_matrix(y_train, y_train_pred))


## 10. Validation Metrics

In [ ]:
y_val_pred = lgbm_model.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Validation Weighted Precision:", precision_score(y_val, y_val_pred, average="weighted", zero_division=0))
print("Validation Weighted Recall:", recall_score(y_val, y_val_pred, average="weighted", zero_division=0))
print("Validation Weighted F1:", f1_score(y_val, y_val_pred, average="weighted", zero_division=0))
print("\nValidation Classification Report:\n")
print(classification_report(y_val, y_val_pred, target_names=target_le.classes_))
print("Validation Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))


## 11. Testing Metrics

In [ ]:
y_test_pred = lgbm_model.predict(X_test)

print("Testing Accuracy:", accuracy_score(y_test, y_test_pred))
print("Testing Weighted Precision:", precision_score(y_test, y_test_pred, average="weighted", zero_division=0))
print("Testing Weighted Recall:", recall_score(y_test, y_test_pred, average="weighted", zero_division=0))
print("Testing Weighted F1:", f1_score(y_test, y_test_pred, average="weighted", zero_division=0))
print("\nTesting Classification Report:\n")
print(classification_report(y_test, y_test_pred, target_names=target_le.classes_))
print("Testing Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))


## 12. Feature Importance

In [ ]:
feature_importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": lgbm_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance_df.head(20)


In [ ]:
plt.figure(figsize=(10, 6))
top_feat = feature_importance_df.head(15).sort_values(by="Importance")
plt.barh(top_feat["Feature"], top_feat["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 15 Feature Importances - Health Risk Label (LightGBM)")
plt.show()


## 13. Save Outputs

In [ ]:
model_df.iloc[train_idx].to_csv("health_train_70.csv", index=False)
model_df.iloc[temp_idx].iloc[val_idx_rel].to_csv("health_validation_20.csv", index=False)
model_df.iloc[temp_idx].iloc[test_idx_rel].to_csv("health_test_10.csv", index=False)

joblib.dump(lgbm_model, "health_risk_lightgbm_model.joblib")
joblib.dump(target_le, "health_risk_label_encoder.joblib")
joblib.dump(gender_le, "health_gender_encoder.joblib")

train_pred_df = X_train.copy()
train_pred_df["actual_risk_label"] = target_le.inverse_transform(y_train)
train_pred_df["predicted_risk_label"] = target_le.inverse_transform(y_train_pred)
train_pred_df.to_csv("health_train_predictions.csv", index=False)

val_pred_df = X_val.copy()
val_pred_df["actual_risk_label"] = target_le.inverse_transform(y_val)
val_pred_df["predicted_risk_label"] = target_le.inverse_transform(y_val_pred)
val_pred_df.to_csv("health_validation_predictions.csv", index=False)

test_pred_df = X_test.copy()
test_pred_df["actual_risk_label"] = target_le.inverse_transform(y_test)
test_pred_df["predicted_risk_label"] = target_le.inverse_transform(y_test_pred)
test_pred_df.to_csv("health_test_predictions.csv", index=False)

print("All outputs saved successfully.")
